# Learning Objectives

* Build a small fan-out / fan-in Pegasus workflow from Python wrapper scripts and an Apptainer container.
* Fetch public data at run time from inside a job (GBIF occurrence API) instead of registering inputs up front.
* Visualize a workflow DAG before submitting it.
* Plan, submit, monitor, and pull statistics for a Pegasus workflow from a notebook.
* Compare where and when several species have been recorded.

# Species Occurrence Comparison Workflow

This workflow compares where and when several species have been recorded, using public
[GBIF](https://www.gbif.org/) occurrence data. It was written spec-first: `SPEC.md` was reviewed before any
code, and every constraint in the generator traces back to it. Each species is fetched and summarised
independently, so Pegasus runs the species in parallel; one merge job then builds the comparison.
**No API key or account is required.** See `README.md` for the full description.

**Workflow Jobs:**
1. `fetch_occurrences` (per species) - resolve the name against the GBIF backbone and download up to `--max-records` georeferenced records
2. `summarise_species` (per species) - record count, latitude range and median, year range, top countries, and a coordinate scatter map
3. `merge_species` (once) - one comparison table and chart across all species

**Prerequisites:** [Pegasus WMS](https://pegasus.isi.edu/) >= 5.0, [HTCondor](https://htcondor.org/) >= 10.2,
Python 3.8+ with `pyyaml`, and [Apptainer](https://apptainer.org/) — all assumed to already be available in
this notebook's environment. Worker nodes need outbound HTTPS to `api.gbif.org`.

## 1. Input Data

There is nothing to prepare: each `fetch_occurrences` job downloads its species' records from the GBIF API
when it runs. Choose the species and the year range here.

In [ ]:
SPECIES = ["Danaus plexippus", "Vanessa cardui", "Apis mellifera"]
YEAR_FROM, YEAR_TO = 2015, 2024
MAX_RECORDS = 900

## 2. Build the Apptainer Container

The workflow's jobs run inside an Apptainer container (`Apptainer/Species_Container.def`) that bundles
Python 3 with pandas and matplotlib. Building it can take a few minutes; skip the rebuild if the `.sif`
already exists.

In [ ]:
%%bash
if [ ! -f Apptainer/Species_Container.sif ]; then
    apptainer build Apptainer/Species_Container.sif Apptainer/Species_Container.def
else
    echo "Container already built: Apptainer/Species_Container.sif"
fi

## 3. Generate the Workflow

The following cell uses the `SpeciesOccurrenceWorkflow` class from `workflow_generator.py` — the same
generator used by the command-line tool — to build the Pegasus workflow DAG (site, transformation, and
replica catalogs) and writes it to `workflow.yml`. Nothing about the workflow is defined here: change the
pipeline in `workflow_generator.py`, and both the notebook and the CLI pick it up.

`create_sites_catalog()` writes a self-contained site catalog for this machine's HTCondor pool (`local` +
`compute`). To run on a resource with a centrally hosted site catalog
([pegasus-site-catalogs](https://github.com/pegasushub/pegasus-site-catalogs/tree/main/conf), e.g.
`access-pegasus.yml`, `unity.yml`), set `HOSTED_SITE_CATALOG` instead — or set it once in `~/.pegasusrc`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from workflow_generator import SpeciesOccurrenceWorkflow

DAGFILE = "workflow.yml"
EXEC_SITE = "compute"
HOSTED_SITE_CATALOG = None  # e.g. "unity.yml"; None = local HTCondor catalog below

workflow = SpeciesOccurrenceWorkflow(
    dagfile=DAGFILE, container_sif="Apptainer/Species_Container.sif"
)

print("Creating workflow properties...")
workflow.create_pegasus_properties(hosted_site_catalog=HOSTED_SITE_CATALOG)

if HOSTED_SITE_CATALOG is None:
    print("Creating execution sites...")
    workflow.create_sites_catalog(exec_site_name=EXEC_SITE)

print("Creating transformation catalog...")
workflow.create_transformation_catalog(exec_site_name=EXEC_SITE)

print("Creating replica catalog...")
workflow.create_replica_catalog()

print("Creating species occurrence workflow DAG...")
workflow.create_workflow(SPECIES, YEAR_FROM, YEAR_TO, MAX_RECORDS)

workflow.write()
print("\nSpecies Occurrence Workflow has been generated!")

## 4. View the Generated Workflow DAG

Before submitting, visualize the workflow DAG using `pegasus-graphviz`. Each species is an independent
`fetch -> summarise` chain; all of them feed the single `merge_all` job.

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image

Image(filename="workflow.png")

## 5. Plan and Submit the Workflow

The workflow will be planned and submitted for execution on the `compute` site defined in the site catalog
chosen above.

The CLI version of this generator (`workflow_generator.py`) writes the same workflow non-interactively and
prints the `pegasus-plan` command; it never submits by itself. Submitting is this explicit step.

In [ ]:
workflow.plan_submit(exec_site_name=EXEC_SITE)

## Workflow Status Monitoring

After successful submission, monitor the workflow status. The output shows job counts and idle/running/completed
states. `wait()` blocks (with a progress bar) until the workflow completes or fails.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 6. Statistics

Once the workflow completes, pull execution statistics from the Pegasus provenance database.

In [ ]:
workflow.statistics()

## 7. Examine the Results

The workflow stages the following final outputs to `output/`. Raw occurrence files stay on the execution
side and are not staged back (SPEC C7).

| Output | Description |
|--------|-------------|
| `<species>_stats.json` | Record count, latitude range and median, year range, top countries, taxon key and match confidence |
| `<species>_map.png` | Scatter of record coordinates |
| `comparison.csv` | One row per requested species |
| `comparison.png` | Records per species, and latitudinal spread |

In [ ]:
!ls -ltR output/

The comparison across species. Occurrence counts reflect observer effort, not abundance (SPEC N3):

In [ ]:
import pandas as pd

display(pd.read_csv("output/comparison.csv"))
Image(filename="output/comparison.png")